In [ ]:
# === 노트북 공통 preamble (llm_math 패키지 로드) ===
import sys
from pathlib import Path

# llm_math 패키지를 찾을 수 있는 후보 경로들
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 현재 디렉토리의 상위 디렉토리들도 후보에 추가 (notebooks/ 폴더에서 실행 시)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math import 시도
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[주의] llm_math 패키지 로드 실패: {e}")
    print("  GitHub 레포를 클론하고 colab_setup.sh를 실행하세요.")
# === preamble 끝 ===


# Ch 25. 양자화 — 모델을 작고 가볍게 [CPU/GPU 벤치마크 ⑪]

> **학습 목표**
> - FP32, FP16, BF16, INT8, INT4의 정밀도·메모리·속도 트레이드오프를 이해한다
> - 양자화 수학 $q = \mathrm{round}(x/s) + z$를 유도한다
> - GPTQ, AWQ, GGUF 양자화 기법의 차이를 안다

## 25.1 왜 양자화인가

LLM은 메모리·연산량이 큼:
- LLaMA-7B FP16: 14GB
- 추론 시 추가 KV Cache
- 모바일/엣지 배포 어려움

양자화로 해결:
- FP16 → INT8: 메모리 절반, 속도 2배
- FP16 → INT4: 메모리 1/4, 속도 3-4배

## 25.2 부동소수점 표현

| 타입 | 부호 | 지수 | 가수 | 총 |
|---|---|---|---|---|
| FP32 | 1 | 8 | 23 | 32 bit |
| FP16 | 1 | 5 | 10 | 16 bit |
| BF16 | 1 | 8 | 7 | 16 bit |
| FP8 (E4M3) | 1 | 4 | 3 | 8 bit |

- FP16: 정밀도 좋지만 범위 좁음 (오버플로 위험)
- BF16: FP32와 같은 범위, 정밀도 낮음 → LLM 학습에 적합


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# 부동소수점 범위 비교
print("부동소수Point 타입별 범위:")
for name, dtype in [('FP32', torch.float32), ('FP16', torch.float16), ('BF16', torch.bfloat16)]:
    info = torch.finfo(dtype)
    print(f"  {name}: min={info.min:.2e}, max={info.max:.2e}, eps={info.eps:.2e}")

# FP16 오버플로 예시
print("\nFP16 한계:")
print(f"  2^15 = {float(torch.tensor(2**15, dtype=torch.float16))}")
print(f"  2^16 = {float(torch.tensor(2**16, dtype=torch.float16))}  # inf!")
print(f"  BF16 2^16 = {float(torch.tensor(2**16, dtype=torch.bfloat16))}  # 정상")


## 25.3 정수 양자화 기본

FP32 값을 INT8 (0~255 또는 -128~127)로 변환:

**대칭 양자화** (zero-point = 0):
$$q = \mathrm{round}(x / s), \quad s = \frac{\max|x|}{127}$$
$$\hat{x} = s \cdot q$$

**비대칭 양자화** (zero-point 사용):
$$q = \mathrm{round}(x / s) + z$$
$$s = \frac{\max(x) - \min(x)}{255}, \quad z = -\mathrm{round}(\min(x) / s)$$

양자화 오차: $\epsilon = x - \hat{x}$.


In [ ]:
# 양자화 구현
def quantize_symmetric(x, n_bits=8):
    """Symmetric Quantization."""
    qmax = 2 ** (n_bits - 1) - 1  # 127 for INT8
    scale = x.abs().max() / qmax
    q = torch.round(x / scale).clamp(-qmax - 1, qmax).to(torch.int8)
    return q, scale

def dequantize_symmetric(q, scale):
    return q.to(torch.float32) * scale

# Test
torch.manual_seed(0)
x = torch.randn(100) * 5  # Mean 0, Standard Deviation 5
q, scale = quantize_symmetric(x, n_bits=8)
x_hat = dequantize_symmetric(q, scale)
error = (x - x_hat).abs()
print(f"원본 범위: [{x.min():.3f}, {x.max():.3f}]")
print(f"Quantization 범위: [{q.min()}, {q.max()}]")
print(f"스케일: {scale:.6f}")
print(f"Mean Absolute error: {error.mean():.6f}")
print(f"Maximum Absolute error: {error.max():.6f}")
print(f"상대 Error: {(error.mean() / x.abs().mean() * 100):.3f}%")

# 다양한 비트 수 비교
print("\n비트 수별 Quantization Error:")
for n_bits in [8, 4, 2]:
    q, scale = quantize_symmetric(x, n_bits=n_bits)
    x_hat = dequantize_symmetric(q, scale)
    err = (x - x_hat).abs().mean()
    print(f"  INT{n_bits}: Mean Error = {err:.6f} ({err/x.abs().mean()*100:.2f}%)")


## 25.4 Per-tensor, Per-channel, Per-group

- **Per-tensor**: 전체 텐서에 하나의 scale. 단순하지만 정확도 낮음.
- **Per-channel**: 채널(행/열)마다 scale. 정확도 좋음, 메모리 약간 증가.
- **Per-group**: $g$개 원소마다 scale. 균형점. LLM 양자화 표준.

예: 4-bit 양자화에 group_size=128 → 128개 값마다 scale (FP16) 저장.


In [ ]:
# Per-group 양자화
def quantize_per_group(x, n_bits=4, group_size=128):
    """그룹별 Quantization."""
    qmax = 2 ** (n_bits - 1) - 1
    # 그룹으로 분할
    orig_shape = x.shape
    x_flat = x.reshape(-1, group_size)
    scales = x_flat.abs().max(dim=1, keepdim=True).values / qmax
    q = torch.round(x_flat / scales).clamp(-qmax - 1, qmax)
    return q, scales, orig_shape

def dequantize_per_group(q, scales, orig_shape):
    return (q * scales).reshape(orig_shape)

# Comparison: per-tensor vs per-group
torch.manual_seed(0)
W = torch.randn(256, 256) * 0.1  # Weight Matrix

# per-tensor (4-bit)
qmax = 7  # 4-bit symmetric
scale_t = W.abs().max() / qmax
q_t = torch.round(W / scale_t).clamp(-8, 7)
W_hat_t = q_t * scale_t
err_t = (W - W_hat_t).abs().mean()

# per-group (4-bit, group=128)
q_g, scales_g, shape = quantize_per_group(W, n_bits=4, group_size=128)
W_hat_g = dequantize_per_group(q_g, scales_g, shape)
err_g = (W - W_hat_g).abs().mean()

print(f"4-bit 양자화 오차:")
print(f"  Per-tensor: {err_t:.6f} ({err_t/W.abs().mean()*100:.2f}%)")
print(f"  Per-group (128): {err_g:.6f} ({err_g/W.abs().mean()*100:.2f}%)")
print("\n=> Per-group이 훨씬 정확. LLM Quantization Standard.")


## 25.5 PTQ (Post-Training Quantization) vs QAT

**PTQ**: 학습 후 양자화. 간단하지만 정확도 손실 가능.

**QAT (Quantization-Aware Training)**: 양자화를 시뮬레이션하며 학습. 더 정확하지만 복잡.

LLM은 보통 PTQ 사용 (학습 비용이 너무 큼).

## 25.6 GPTQ, AWQ

### GPTQ
- 2차 오차 최소화 기반
- 층별 양자화 (레이어 하나씩)
- Hessian 행렬 근사로 오차 예측
- INT4에서도 PPL 거의 손실 없음

### AWQ (Activation-aware Weight Quantization)
- 일부 가중치(중요 채널)는 FP16 유지
- 활성값 크기로 중요도 판단
- GPTQ보다 빠르고 단순


## 25.7 QLoRA — 4-bit 양자화 + LoRA

QLoRA (Dettmers et al., 2023):
1. 베이스 모델을 4-bit NF4로 양자화 (고정)
2. LoRA 어댑터만 FP16/BF16으로 학습
3. 메모리: 7B 모델을 단일 24GB GPU에서 파인튜닝 가능

NF4 (NormalFloat 4-bit): 가중치가 정규분포를 따른다는 가정으로 최적화된 4-bit 표현.


In [ ]:
# QLoRA 메모리 추정
def qlora_memory(base_params_b, adapter_ratio=0.01):
    """QLoRA Memory Estimation (GB).
    base_params_b: 베이스 Parameter Count (단위: 10억)
    adapter_ratio: 어댑터 파라미터 Ratio
    """
    # 베이스: 4-bit = 0.5 bytes
    base_mem = base_params_b * 0.5
    # 어댑터: FP16 + AdamW (4배) = 8 bytes
    adapter_mem = base_params_b * adapter_ratio * 8
    # 활성Value (대략)
    activation = 2  # GB
    return base_mem + adapter_mem + activation

# Comparison: 풀 파인튜닝 vs QLoRA
print(f"{'Model':>10} | {'풀 FT FP16 (GB)':>15} | {'QLoRA 4-bit (GB)':>17}")
print("-" * 50)
for name, n_b in [('7B', 7), ('13B', 13), ('30B', 30), ('70B', 70)]:
    full = n_b * 2 * 4 + 4  # FP16 + AdamW FP32 (4배) + 활성
    qlora = qlora_memory(n_b, 0.01)
    print(f"{name:>10} | {full:>15.1f} | {qlora:>17.1f}")
print("\n=> QLoRA로 70B Model도 단일 48GB GPU에서 파인튜닝 가능.")


## 25.8 [CPU/GPU 벤치마크 ⑪] 정밀도별 추론


In [ ]:
# 정밀도별 추론 속도 비교
from llm_math.bench import time_fn

# 간단한 선형층 추론
def bench_linear(d_in, d_out, batch_size, dtype, device='cpu'):
    """LinearLayer Inference Time."""
    model = nn.Linear(d_in, d_out).to(dtype=dtype, device=device)
    x = torch.randn(batch_size, d_in, dtype=dtype, device=device)
    def run():
        with torch.no_grad():
            return model(x)
    return run

print(f"{'dtype':>10} | {'CPU (ms)':>10} | {'Memory (MB)':>12}")
print("-" * 40)
d_in, d_out, bs = 4096, 4096, 8
for name, dtype in [('FP32', torch.float32), ('FP16', torch.float16)]:
    if dtype == torch.float16 and not torch.cuda.is_available():
        # CPU에서 FP16은 제한적
        continue
    try:
        run = bench_linear(d_in, d_out, bs, dtype, 'cpu')
        # warmup
        run()
        res = time_fn(run, device='cpu', warmup=2, repeat=5)
        # Memory (대략)
        mem = d_in * d_out * (4 if dtype == torch.float32 else 2) / 1024**2
        print(f"{name:>10} | {res['mean_ms']:>10.3f} | {mem:>12.2f}")
    except Exception as e:
        print(f"{name:>10} | {'N/A':>10} | error: {e}")

print("\n=> FP16은 Memory 절반, CPU에서는 Speed Improvement 제한적 (GPU에서 큰 Improvement).")


## 25.9 핵심 정리

| 기법 | 비트 | 메모리 절약 | 정확도 손실 |
|---|---|---|---|
| FP16 | 16 | 2x | 거의 없음 |
| BF16 | 16 | 2x | 거의 없음 |
| INT8 | 8 | 4x | 약간 |
| INT4 (GPTQ/AWQ) | 4 | 8x | 작음 |
| QLoRA (NF4) | 4 | 8x | 작음 |

## 연습문제

1. INT8 양자화 후 선형층 출력의 MSE를 측정하라.
2. Per-tensor vs Per-channel vs Per-group 양자화 오차를 비교하라.
3. 그룹 크기 32, 64, 128, 256에 따른 양자화 오차를 비교하라.
4. FP32 vs FP16 추론 속도를 CPU와 GPU에서 각각 비교하라.
5. QLoRA가 풀 파인튜닝과 비교해 메모리를 얼마나 절약하는지 7B, 13B, 70B 모델에 대해 계산하라.

> 해설: `solutions/ch25_solutions.ipynb`
